In [1]:
%pip -q install sentinelhub pandas numpy rasterio python-dateutil tqdm


Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from datetime import timedelta
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from tqdm.auto import tqdm

from sentinelhub import (
    SHConfig, SentinelHubCatalog, DataCollection,
    CRS, BBox, bbox_to_dimensions,
    SentinelHubRequest, MimeType
)

# ----------------------------
# PASTE YOUR CREDENTIALS HERE
# ----------------------------
CLIENT_ID = "sh-a2c7c0a9-4e39-44d9-9c98-e1ad1df57651"
CLIENT_SECRET = "p8v5PEnojoV1WUEtkd4Z70x5QGdya0QR"

config = SHConfig()
config.sh_client_id = CLIENT_ID
config.sh_client_secret = CLIENT_SECRET

# Copernicus Data Space Ecosystem endpoints (required for CDSE OAuth clients)
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
# Define Sentinel-2 L2A collection to use CDSE service URL (critical!)
S2L2A_CDSE = DataCollection.SENTINEL2_L2A.define_from(
    "s2l2a_cdse",
    service_url=config.sh_base_url
)



In [2]:
WINDOW_DAYS = 0.5          # +/- 1 day around SAR center time
MAX_CLOUD = 10           # percent (increase if too few matches)
BASE_RESOLUTION_M = 10   # will auto-coarsen if bbox is too big
MAX_DIM_PX = 2500        # cap output pixel size to avoid oversized requests

OUT_DIR = "dataset_out"
MATCHES_CSV = os.path.join(OUT_DIR, "sar_to_s2_matches_2.csv")
S2_DIR = os.path.join(OUT_DIR, "sentinel2")

BANDS = ["B02","B03","B04","B08","B11","B12"]  # RGB + NIR + SWIR
EVALSCRIPT = f"""
//VERSION=3
function setup() {{
  return {{
    input: [{{
      bands: {BANDS},
      units: "REFLECTANCE"
    }}],
    output: {{
      bands: {len(BANDS)},
      sampleType: "FLOAT32"
    }}
  }};
}}
function evaluatePixel(s) {{
  return [{", ".join([f"s.{b}" for b in BANDS])}];
}}
"""


In [3]:
def pick_resolution_and_size(bbox: BBox, base_res_m: float, max_dim_px: int):
    """Coarsen resolution until bbox fits under max_dim_px."""
    res = float(base_res_m)
    while True:
        w, h = bbox_to_dimensions(bbox, resolution=res)
        if max(w, h) <= max_dim_px:
            return res, (w, h)
        res *= 2

def sar_center_time(date_from: pd.Timestamp, date_to: pd.Timestamp) -> pd.Timestamp:
    return date_from + (date_to - date_from) / 2

def find_best_s2_match(catalog: SentinelHubCatalog, bbox: BBox, center_t: pd.Timestamp,
                       window_days: int, max_cloud: float):
    """
    Return best match dict or None.
    Ranking: smallest |Δt|, then lowest cloud.
    """
    start = (center_t - timedelta(days=window_days)).strftime("%Y-%m-%dT%H:%M:%SZ")
    end   = (center_t + timedelta(days=window_days)).strftime("%Y-%m-%dT%H:%M:%SZ")

    search_iter = catalog.search(
        S2L2A_CDSE, 
        bbox=bbox,
        time=(start, end),
        filter=f"eo:cloud_cover < {max_cloud}",
        fields={"include": ["id", "properties.datetime", "properties.eo:cloud_cover"], "exclude": []},
    )
    items = list(search_iter)
    if not items:
        return None

    def score(item):
        dt = pd.to_datetime(item["properties"]["datetime"], utc=True)
        cloud = float(item["properties"].get("eo:cloud_cover", 9999))
        return (abs((dt - center_t).total_seconds()), cloud)

    best = min(items, key=score)
    best_dt = pd.to_datetime(best["properties"]["datetime"], utc=True)
    best_cloud = float(best["properties"].get("eo:cloud_cover", np.nan))

    return {"s2_id": best["id"], "s2_datetime": best_dt, "s2_cloud": best_cloud}

def download_s2_patch(cfg: SHConfig, bbox: BBox, s2_dt: pd.Timestamp, out_path: str):
    """
    Download multiband patch as GeoTIFF for the bbox around the chosen acquisition time.
    We restrict interval to +/- 2 minutes so Process API picks that acquisition.
    """
    t0 = (s2_dt - pd.Timedelta(minutes=2)).strftime("%Y-%m-%dT%H:%M:%SZ")
    t1 = (s2_dt + pd.Timedelta(minutes=2)).strftime("%Y-%m-%dT%H:%M:%SZ")

    res_m, (width, height) = pick_resolution_and_size(bbox, BASE_RESOLUTION_M, MAX_DIM_PX)

    req = SentinelHubRequest(
        evalscript=EVALSCRIPT,
        input_data=[SentinelHubRequest.input_data(
            data_collection=S2L2A_CDSE,
            time_interval=(t0, t1),
        )],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        bbox=bbox,
        size=(width, height),
        config=cfg,
    )

    arr = req.get_data()[0]  # (H, W, C)

    minx, miny, maxx, maxy = bbox
    transform = from_bounds(minx, miny, maxx, maxy, width, height)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    profile = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": arr.shape[2],
        "dtype": str(arr.dtype),
        "crs": "EPSG:4326",
        "transform": transform,
    }

    with rasterio.open(out_path, "w", **profile) as dst:
        for i in range(arr.shape[2]):
            dst.write(arr[:, :, i], i + 1)


In [4]:
csv_path = "DARTIS_2019_bboxes_for_sentinel.csv"  
df = pd.read_csv(csv_path)
df.head()


,min_lon,min_lat,max_lon,max_lat,date_from,date_to,sentinel_id,patch_name,image_file,label_tag
0,32.986494,33.191126,33.144526,33.325226,2019-01-01T03:42:35Z,2019-01-01T03:43:50Z,S1B_IW_GRDH_1SDV_20190101T034300_20190101T0343...,S1_20190101_034235_034350_VV_1,ow-0001.jpg,ow-0001-01-000001
1,31.949329,31.619308,32.106127,31.754192,2019-01-04T15:56:38Z,2019-01-04T15:58:18Z,S1A_IW_GRDH_1SDV_20190104T155703_20190104T1557...,S1_20190104_155638_155818_VV_2,ow-0002.jpg,ow-0002-01-000002
2,30.548837,31.505394,30.705499,31.640837,2019-01-10T15:56:11Z,2019-01-10T15:56:35Z,S1B_IW_GRDH_1SDV_20190110T155611_20190110T1556...,S1_20190110_155611_155635_VV_9,ow-0003.jpg,ow-0003-01-000003
3,31.104025,31.644936,31.261357,31.780146,2019-01-10T15:56:11Z,2019-01-10T15:56:35Z,S1B_IW_GRDH_1SDV_20190110T155611_20190110T1556...,S1_20190110_155611_155635_VV_10,ow-0004.jpg,ow-0004-01-000004
4,31.104025,31.644936,31.261357,31.780146,2019-01-10T15:56:11Z,2019-01-10T15:56:35Z,S1B_IW_GRDH_1SDV_20190110T155611_20190110T1556...,S1_20190110_155611_155635_VV_10,ow-0004.jpg,ow-0004-02-000005


In [6]:
import os, time, random
from concurrent.futures import ThreadPoolExecutor, as_completed

os.makedirs(OUT_DIR, exist_ok=True)

# parse times
df["date_from"] = pd.to_datetime(df["date_from"], utc=True)
df["date_to"] = pd.to_datetime(df["date_to"], utc=True)
df["sar_center_time"] = df.apply(lambda r: sar_center_time(r["date_from"], r["date_to"]), axis=1)

MAX_WORKERS = 8  
RETRIES = 4

def _match_one_row(r):
    """
    r is a dict-like row. Returns a dict for matches[] or None.
    """
    bbox = BBox((r["min_lon"], r["min_lat"], r["max_lon"], r["max_lat"]), crs=CRS.WGS84)
    center_t = r["sar_center_time"]

    # Create a local client per thread to avoid thread-safety issues.
    catalog_local = SentinelHubCatalog(config=config)

    last_err = None
    for k in range(RETRIES):
        try:
            m = find_best_s2_match(catalog_local, bbox, center_t, WINDOW_DAYS, MAX_CLOUD)
            if m is None:
                return None  # no match
            return {
                "patch_name": r["patch_name"],
                "image_file": r.get("image_file", None),
                "label_tag": r.get("label_tag", None),
                "sentinel1_id": r.get("sentinel_id", None),
                "sar_date_from": r["date_from"].strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sar_date_to": r["date_to"].strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sar_center_time": center_t.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "min_lon": r["min_lon"], "min_lat": r["min_lat"],
                "max_lon": r["max_lon"], "max_lat": r["max_lat"],
                "s2_id": m["s2_id"],
                "s2_datetime": m["s2_datetime"].strftime("%Y-%m-%dT%H:%M:%SZ"),
                "s2_cloud": m["s2_cloud"],
            }
        except Exception as e:
            last_err = e
            # exponential backoff + jitter
            sleep_s = (2 ** k) * 0.5 + random.random() * 0.25
            time.sleep(sleep_s)

    # If all retries failed, you can either return None or surface the error
    # Returning None keeps the pipeline moving.
    print("Failed row:", r.get("patch_name"), "error:", repr(last_err))
    return None

# Convert to dict records so threads don't touch the dataframe internals
records = df.to_dict("records")

matches = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(_match_one_row, r) for r in records]
    for fut in tqdm(as_completed(futures), total=len(futures)):
        res = fut.result()
        if res is not None:
            matches.append(res)

matches_df = pd.DataFrame(matches)
matches_df.to_csv(MATCHES_CSV, index=False)
print(f"Saved matches: {len(matches_df)} rows -> {MATCHES_CSV}")

matches_df.head()


  0%|          | 0/5515 [00:00<?, ?it/s]

Saved matches: 778 rows -> dataset_out\sar_to_s2_matches_2.csv


,patch_name,image_file,label_tag,sentinel1_id,sar_date_from,sar_date_to,sar_center_time,min_lon,min_lat,max_lon,max_lat,s2_id,s2_datetime,s2_cloud
0,S1_20190104_155638_155818_VV_2,ow-0002.jpg,ow-0002-01-000002,S1A_IW_GRDH_1SDV_20190104T155703_20190104T1557...,2019-01-04T15:56:38Z,2019-01-04T15:58:18Z,2019-01-04T15:57:28Z,31.949329,31.619308,32.106127,31.754192,S2A_MSIL2A_20190104T083331_N0500_R021_T36RUV_2...,2019-01-04T08:41:31Z,0.00
1,S1_20190112_035117_035259_VV_0,ow-0007.jpg,ow-0007-01-000014,S1A_IW_GRDH_1SDV_20190112T035232_20190112T0352...,2019-01-12T03:51:17Z,2019-01-12T03:52:59Z,2019-01-12T03:52:08Z,31.103129,31.596094,31.259670,31.731046,S2B_MSIL2A_20190112T084319_N0500_R064_T36SUA_2...,2019-01-12T08:51:18Z,1.45
2,S1_20190122_155641_155735_VV_0,ow-0015.jpg,ow-0015-01-000032,S1B_IW_GRDH_1SDV_20190122T155641_20190122T1557...,2019-01-22T15:56:41Z,2019-01-22T15:57:35Z,2019-01-22T15:57:08Z,31.160483,32.720001,31.533496,33.037230,S2B_MSIL2A_20190122T084249_N0500_R064_T36SUB_2...,2019-01-22T08:51:04Z,3.89
3,S1_20190122_155641_155735_VV_2,ow-0016.jpg,ow-0016-01-000034,S1B_IW_GRDH_1SDV_20190122T155641_20190122T1557...,2019-01-22T15:56:41Z,2019-01-22T15:57:35Z,2019-01-22T15:57:08Z,30.195389,33.563524,30.361045,33.703140,S2B_MSIL2A_20190122T084249_N0500_R064_T35SQT_2...,2019-01-22T08:50:56Z,3.23
4,S1_20190122_155641_155735_VV_2,ow-0016.jpg,ow-0016-02-000035,S1B_IW_GRDH_1SDV_20190122T155641_20190122T1557...,2019-01-22T15:56:41Z,2019-01-22T15:57:35Z,2019-01-22T15:57:08Z,30.195389,33.563524,30.361045,33.703140,S2B_MSIL2A_20190122T084249_N0500_R064_T35SQT_2...,2019-01-22T08:50:56Z,3.23


##  donwload  sent 2 as png 

In [5]:
import io
import numpy as np
from PIL import Image  # Pillow


from sentinelhub import SentinelHubRequest, MimeType

def download_s2_patch_png(cfg, bbox, s2_dt, out_path,
                          rgb_bands=("B04","B03","B02"),
                          clip=(0.0, 0.3)):
    """
    Downloads an RGB PNG (8-bit) scaled from reflectance.
    clip: reflectance range mapped to [0,255]; adjust if too dark/bright.
    """
    t0 = (s2_dt - pd.Timedelta(minutes=2)).strftime("%Y-%m-%dT%H:%M:%SZ")
    t1 = (s2_dt + pd.Timedelta(minutes=2)).strftime("%Y-%m-%dT%H:%M:%SZ")

    res_m, (width, height) = pick_resolution_and_size(bbox, BASE_RESOLUTION_M, MAX_DIM_PX)

    evalscript_rgb = f"""
    //VERSION=3
    function setup() {{
      return {{
        input: [{{
          bands: ["{rgb_bands[0]}","{rgb_bands[1]}","{rgb_bands[2]}"],
          units: "REFLECTANCE"
        }}],
        output: {{
          bands: 3,
          sampleType: "FLOAT32"
        }}
      }};
    }}
    function evaluatePixel(s) {{
      return [s.{rgb_bands[0]}, s.{rgb_bands[1]}, s.{rgb_bands[2]}];
    }}
    """

    req = SentinelHubRequest(
        evalscript=evalscript_rgb,
        input_data=[SentinelHubRequest.input_data(
            data_collection=S2L2A_CDSE,
            time_interval=(t0, t1),
        )],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],  # we fetch floats then convert
        bbox=bbox,
        size=(width, height),
        config=cfg,
    )

    arr = req.get_data()[0]  # float32 reflectance, shape (H,W,3)

    lo, hi = clip
    arr = np.clip(arr, lo, hi)
    arr = (arr - lo) / (hi - lo + 1e-12)
    img8 = (arr * 255.0).astype(np.uint8)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    Image.fromarray(img8, mode="RGB").save(out_path, format="PNG")


In [6]:
import os, time, random
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

os.makedirs(S2_DIR, exist_ok=True)

matches_df = pd.read_csv(MATCHES_CSV)
matches_df["s2_datetime"] = pd.to_datetime(matches_df["s2_datetime"], utc=True)


matches_df["_key"] = matches_df["patch_name"].astype(str) + "||" + matches_df["s2_datetime"].astype(str)
matches_df = matches_df.drop_duplicates("_key").reset_index(drop=True)
# matches_df = matches_df.drop_duplicates("_key").head(50).reset_index(drop=True)# Keep only 50

MAX_WORKERS = 2
RETRIES = 2

def _download_one(r):
    bbox = BBox((r["min_lon"], r["min_lat"], r["max_lon"], r["max_lat"]), crs=CRS.WGS84)
    s2_dt = r["s2_datetime"]

    out_name = f'{r["patch_name"]}__S2__{s2_dt.strftime("%Y%m%dT%H%M%S")}Z.png'
    out_path = os.path.join(S2_DIR, out_name)

    last_err = None
    for k in range(RETRIES):
        try:
            download_s2_patch_png(config, bbox, s2_dt, out_path)
            return out_path
        except Exception as e:
            last_err = e
            time.sleep((2 ** k) * 0.5 + random.random() * 0.25)

    return ("FAILED", r["patch_name"], str(last_err))

records = matches_df.to_dict("records")

downloaded = []
failed = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(_download_one, r) for r in records]
    for fut in tqdm(as_completed(futures), total=len(futures)):
        res = fut.result()
        if isinstance(res, tuple) and res and res[0] == "FAILED":
            failed.append(res)
        else:
            downloaded.append(res)

print(f"Downloaded {len(downloaded)} PNGs into {S2_DIR}")
if failed:
    print(f"Failed: {len(failed)} (showing up to 5)")
    for x in failed[:5]:
        print(x)


  0%|          | 0/521 [00:00<?, ?it/s]

c:\Users\nurla\miniconda3\envs\notebooks\Lib\site-packages\sentinelhub\download\sentinelhub_client.py:93: SHRateLimitWarning: Download rate limit hit
  warnings.warn("Download rate limit hit", category=SHRateLimitWarning)
c:\Users\nurla\miniconda3\envs\notebooks\Lib\site-packages\sentinelhub\download\sentinelhub_client.py:93: SHRateLimitWarning: Download rate limit hit
  warnings.warn("Download rate limit hit", category=SHRateLimitWarning)
c:\Users\nurla\miniconda3\envs\notebooks\Lib\site-packages\sentinelhub\download\sentinelhub_client.py:93: SHRateLimitWarning: Download rate limit hit
  warnings.warn("Download rate limit hit", category=SHRateLimitWarning)
c:\Users\nurla\miniconda3\envs\notebooks\Lib\site-packages\sentinelhub\download\sentinelhub_client.py:93: SHRateLimitWarning: Download rate limit hit
  warnings.warn("Download rate limit hit", category=SHRateLimitWarning)
c:\Users\nurla\miniconda3\envs\notebooks\Lib\site-packages\sentinelhub\download\sentinelhub_client.py:93: SHRate

Downloaded 521 PNGs into dataset_out\sentinel2


## sar

In [13]:
import os, time, random
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from PIL import Image

# CDSE Sentinel-1 IW GRD collection (service_url must be CDSE base URL)
S1IW_CDSE = DataCollection.SENTINEL1_IW.define_from(
    "s1iw_cdse",
    service_url=config.sh_base_url
)

def download_s1_patch_png(cfg, bbox, t_from, t_to, out_path,
                          pol="VV",
                          p_low=2, p_high=98,
                          gamma=0.9):
    """
    Brighter SAR PNG via log compression + percentile stretch.

    - Convert LINEAR_POWER -> log1p (dB-like compression)
    - Stretch to [0,1] using percentiles (p_low, p_high)
    - Optional gamma (<1 makes brighter)
    """
    evalscript_s1 = f"""
    //VERSION=3
    function setup() {{
      return {{
        input: [{{
          bands: ["{pol}"],
          units: "LINEAR_POWER"
        }}],
        output: {{
          bands: 1,
          sampleType: "FLOAT32"
        }}
      }};
    }}
    function evaluatePixel(s) {{
      return [s.{pol}];
    }}
    """

    res_m, (width, height) = pick_resolution_and_size(bbox, BASE_RESOLUTION_M, MAX_DIM_PX)

    req = SentinelHubRequest(
        evalscript=evalscript_s1,
        input_data=[SentinelHubRequest.input_data(
            data_collection=S1IW_CDSE,
            time_interval=(t_from, t_to),
            other_args={"processing": {"orthorectify": "true"}}
        )],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        bbox=bbox,
        size=(width, height),
        config=cfg,
    )

    arr = req.get_data()[0]

    # handle both (H,W) and (H,W,1)
    img = arr[:, :, 0] if arr.ndim == 3 else arr
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

    # ---- NEW: log compression ----
    img = np.log1p(np.maximum(img, 0.0))  # log(1 + x)

    # ---- NEW: percentile stretch ----
    lo = np.percentile(img, p_low)
    hi = np.percentile(img, p_high)
    if hi <= lo + 1e-12:
        lo, hi = float(img.min()), float(img.max() + 1e-6)

    img = np.clip(img, lo, hi)
    img = (img - lo) / (hi - lo + 1e-12)

    # ---- NEW: gamma (brighter if gamma < 1) ----
    if gamma is not None:
        img = np.clip(img, 0, 1) ** float(gamma)

    img8 = (img * 255.0).astype(np.uint8)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    Image.fromarray(img8, mode="L").save(out_path, format="PNG")

In [6]:
import pandas as pd

S1_DIR = os.path.join(OUT_DIR, "sent1")
COMBINED_DIR = os.path.join(OUT_DIR, "combined")
os.makedirs(S1_DIR, exist_ok=True)
os.makedirs(COMBINED_DIR, exist_ok=True)

matches_df = pd.read_csv(MATCHES_CSV)
matches_df["s2_datetime"] = pd.to_datetime(matches_df["s2_datetime"], utc=True)
matches_df["sar_date_from"] = pd.to_datetime(matches_df["sar_date_from"], utc=True)
matches_df["sar_date_to"] = pd.to_datetime(matches_df["sar_date_to"], utc=True)

# same "unique then first 50" rule
matches_df["_key"] = matches_df["patch_name"].astype(str) + "||" + matches_df["s2_datetime"].astype(str)
sel = matches_df.drop_duplicates("_key").reset_index(drop=True)

print("Selected rows:", len(sel))
sel[["patch_name","sar_date_from","sar_date_to","s2_datetime","s2_cloud"]].head()


Selected rows: 521


,patch_name,sar_date_from,sar_date_to,s2_datetime,s2_cloud
0,S1_20190104_155638_155818_VV_2,2019-01-04 15:56:38+00:00,2019-01-04 15:58:18+00:00,2019-01-04 08:41:31+00:00,0.00
1,S1_20190112_035117_035259_VV_0,2019-01-12 03:51:17+00:00,2019-01-12 03:52:59+00:00,2019-01-12 08:51:18+00:00,1.45
2,S1_20190122_155641_155735_VV_0,2019-01-22 15:56:41+00:00,2019-01-22 15:57:35+00:00,2019-01-22 08:51:04+00:00,3.89
3,S1_20190122_155641_155735_VV_2,2019-01-22 15:56:41+00:00,2019-01-22 15:57:35+00:00,2019-01-22 08:50:56+00:00,3.23
4,S1_20190122_155641_155735_VV_3,2019-01-22 15:56:41+00:00,2019-01-22 15:57:35+00:00,2019-01-22 08:50:56+00:00,3.23


In [11]:
MAX_WORKERS = 2
RETRIES = 4

def _download_one_s1(r):
    bbox = BBox((r["min_lon"], r["min_lat"], r["max_lon"], r["max_lat"]), crs=CRS.WGS84)

    # Use the SAR acquisition interval from original CSV (already stored in matches csv)
    t_from = pd.to_datetime(r["sar_date_from"], utc=True).strftime("%Y-%m-%dT%H:%M:%SZ")
    t_to   = pd.to_datetime(r["sar_date_to"], utc=True).strftime("%Y-%m-%dT%H:%M:%SZ")

    out_name = f'{r["patch_name"]}__S1__{pd.to_datetime(r["sar_date_from"], utc=True).strftime("%Y%m%dT%H%M%S")}Z.png'
    out_path = os.path.join(S1_DIR, out_name)

    last_err = None
    for k in range(RETRIES):
        try:
            download_s1_patch_png(config, bbox, t_from, t_to, out_path, pol="VV", p_low=2, p_high=98, gamma=0.8)
            return out_path
        except Exception as e:
            last_err = e
            time.sleep((2 ** k) * 0.5 + random.random() * 0.25)

    return ("FAILED", r["patch_name"], str(last_err))

records = sel.to_dict("records")

downloaded_s1, failed_s1 = [], []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(_download_one_s1, r) for r in records]
    for fut in tqdm(as_completed(futures), total=len(futures)):
        res = fut.result()
        if isinstance(res, tuple) and res and res[0] == "FAILED":
            failed_s1.append(res)
        else:
            downloaded_s1.append(res)

print(f"S1 downloaded: {len(downloaded_s1)} into {S1_DIR}")
if failed_s1:
    print("S1 failures (up to 5):")
    for x in failed_s1[:5]:
        print(x)


  0%|          | 0/521 [00:00<?, ?it/s]

c:\Users\nurla\miniconda3\envs\notebooks\Lib\site-packages\sentinelhub\download\sentinelhub_client.py:93: SHRateLimitWarning: Download rate limit hit
  warnings.warn("Download rate limit hit", category=SHRateLimitWarning)
c:\Users\nurla\miniconda3\envs\notebooks\Lib\site-packages\sentinelhub\download\sentinelhub_client.py:93: SHRateLimitWarning: Download rate limit hit
  warnings.warn("Download rate limit hit", category=SHRateLimitWarning)
c:\Users\nurla\miniconda3\envs\notebooks\Lib\site-packages\sentinelhub\download\sentinelhub_client.py:93: SHRateLimitWarning: Download rate limit hit
  warnings.warn("Download rate limit hit", category=SHRateLimitWarning)
c:\Users\nurla\miniconda3\envs\notebooks\Lib\site-packages\sentinelhub\download\sentinelhub_client.py:93: SHRateLimitWarning: Download rate limit hit
  warnings.warn("Download rate limit hit", category=SHRateLimitWarning)
c:\Users\nurla\miniconda3\envs\notebooks\Lib\site-packages\sentinelhub\download\sentinelhub_client.py:93: SHRate

S1 downloaded: 521 into dataset_out\sent1


## combining images 


In [12]:
def s2_png_path(row):
    s2_dt = pd.to_datetime(row["s2_datetime"], utc=True)
    out_name = f'{row["patch_name"]}__S2__{s2_dt.strftime("%Y%m%dT%H%M%S")}Z.png'
    return os.path.join(S2_DIR, out_name)

def s1_png_path(row):
    s1_from = pd.to_datetime(row["sar_date_from"], utc=True)
    out_name = f'{row["patch_name"]}__S1__{s1_from.strftime("%Y%m%dT%H%M%S")}Z.png'
    return os.path.join(S1_DIR, out_name)

combined = sel.copy()
combined["s1_file"] = combined.apply(s1_png_path, axis=1)
combined["s2_file"] = combined.apply(s2_png_path, axis=1)

combined["s1_time"] = pd.to_datetime(combined["sar_date_from"], utc=True)
combined["s2_time"] = pd.to_datetime(combined["s2_datetime"], utc=True)
combined["dt_hours"] = (combined["s2_time"] - combined["s1_time"]).dt.total_seconds() / 3600.0

# Keep only rows where both files exist
combined["s1_exists"] = combined["s1_file"].apply(os.path.exists)
combined["s2_exists"] = combined["s2_file"].apply(os.path.exists)
combined_ok = combined[combined["s1_exists"] & combined["s2_exists"]].copy()

out_csv = os.path.join(COMBINED_DIR, "combined_pairs.csv")
combined_ok.to_csv(out_csv, index=False)

print("Combined rows:", len(combined_ok))
print("Saved:", out_csv)
combined_ok[["patch_name","s1_time","s2_time","dt_hours","s1_file","s2_file"]].head()


Combined rows: 521
Saved: dataset_out\combined\combined_pairs.csv


,patch_name,s1_time,s2_time,dt_hours,s1_file,s2_file
0,S1_20190104_155638_155818_VV_2,2019-01-04 15:56:38+00:00,2019-01-04 08:41:31+00:00,-7.251944,dataset_out\sent1\S1_20190104_155638_155818_VV...,dataset_out\sentinel2\S1_20190104_155638_15581...
1,S1_20190112_035117_035259_VV_0,2019-01-12 03:51:17+00:00,2019-01-12 08:51:18+00:00,5.000278,dataset_out\sent1\S1_20190112_035117_035259_VV...,dataset_out\sentinel2\S1_20190112_035117_03525...
2,S1_20190122_155641_155735_VV_0,2019-01-22 15:56:41+00:00,2019-01-22 08:51:04+00:00,-7.093611,dataset_out\sent1\S1_20190122_155641_155735_VV...,dataset_out\sentinel2\S1_20190122_155641_15573...
3,S1_20190122_155641_155735_VV_2,2019-01-22 15:56:41+00:00,2019-01-22 08:50:56+00:00,-7.095833,dataset_out\sent1\S1_20190122_155641_155735_VV...,dataset_out\sentinel2\S1_20190122_155641_15573...
4,S1_20190122_155641_155735_VV_3,2019-01-22 15:56:41+00:00,2019-01-22 08:50:56+00:00,-7.095833,dataset_out\sent1\S1_20190122_155641_155735_VV...,dataset_out\sentinel2\S1_20190122_155641_15573...


In [ ]:
from pathlib import Path
import os
import pandas as pd

def _rel(p, base):
    return Path(os.path.relpath(p, base)).as_posix()

html_rows = []
base = COMBINED_DIR

for _, r in combined_ok.iterrows():
    s1_rel = _rel(r["s1_file"], base)
    s2_rel = _rel(r["s2_file"], base)

    s1_ts = pd.to_datetime(r["s1_time"], utc=True).strftime("%Y-%m-%d %H:%M:%S UTC")
    s2_ts = pd.to_datetime(r["s2_time"], utc=True).strftime("%Y-%m-%d %H:%M:%S UTC")
    dt_h  = float(r["dt_hours"])

   
    img_name = str(r.get("image_file", "") or "")

    html_rows.append(f"""
    <div style="display:flex; align-items:center; gap:24px; margin:18px 0;">
      <div style="text-align:center;">
        <div style="font-family:monospace; margin-bottom:6px;">S1: {s1_ts}</div>
        <img src="{s1_rel}" style="max-width:320px; border:1px solid #ccc;" />
      </div>

      <div style="text-align:center; font-family:monospace; min-width:220px;">
        <div style="font-size:18px; margin-bottom:6px;">Δt = {dt_h:.2f} h</div>
        <div style="font-size:16px; color:#333;">{img_name}</div>
      </div>

      <div style="text-align:center;">
        <div style="font-family:monospace; margin-bottom:6px;">S2: {s2_ts}</div>
        <img src="{s2_rel}" style="max-width:320px; border:1px solid #ccc;" />
      </div>
    </div>
    <hr/>
    """)

html = f"""
<html><head><meta charset="utf-8"><title>S1–S2 pairs</title></head>
<body>
<h2>S1–S2 pairs :oc : oil/coast; ow : oil/water; nc : no_oil/coast; nw : no_oil/water</h2>
{''.join(html_rows)}
</body></html>
"""

preview_path = os.path.join(COMBINED_DIR, "preview.html")
with open(preview_path, "w", encoding="utf-8") as f:
    f.write(html)

print("Wrote:", preview_path)
print("Open preview.html in your browser.")


Wrote: dataset_out\combined\preview.html
Open preview.html in your browser.


## tiff download 

In [7]:
import os, time, random
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

S1_TIFF_DIR = os.path.join(OUT_DIR, "sent1_tiff")
S2_TIFF_DIR = os.path.join(OUT_DIR, "sent2_tiff")
os.makedirs(S1_TIFF_DIR, exist_ok=True)
os.makedirs(S2_TIFF_DIR, exist_ok=True)

COMBINED_CSV = os.path.join(OUT_DIR, "combined", "combined_pairs.csv")
pairs = pd.read_csv(COMBINED_CSV)

# parse timestamps
pairs["s1_time"] = pd.to_datetime(pairs["s1_time"], utc=True)
pairs["s2_time"] = pd.to_datetime(pairs["s2_time"], utc=True)
pairs["sar_date_from"] = pd.to_datetime(pairs["sar_date_from"], utc=True)
pairs["sar_date_to"] = pd.to_datetime(pairs["sar_date_to"], utc=True)

print("Rows in combined_pairs.csv:", len(pairs))
pairs.head()


Rows in combined_pairs.csv: 521


,patch_name,image_file,label_tag,sentinel1_id,sar_date_from,sar_date_to,sar_center_time,min_lon,min_lat,max_lon,...,s2_datetime,s2_cloud,_key,s1_file,s2_file,s1_time,s2_time,dt_hours,s1_exists,s2_exists
0,S1_20190104_155638_155818_VV_2,ow-0002.jpg,ow-0002-01-000002,S1A_IW_GRDH_1SDV_20190104T155703_20190104T1557...,2019-01-04 15:56:38+00:00,2019-01-04 15:58:18+00:00,2019-01-04T15:57:28Z,31.949329,31.619308,32.106127,...,2019-01-04 08:41:31+00:00,0.00,S1_20190104_155638_155818_VV_2||2019-01-04 08:...,dataset_out\sent1\S1_20190104_155638_155818_VV...,dataset_out\sentinel2\S1_20190104_155638_15581...,2019-01-04 15:56:38+00:00,2019-01-04 08:41:31+00:00,-7.251944,True,True
1,S1_20190112_035117_035259_VV_0,ow-0007.jpg,ow-0007-01-000014,S1A_IW_GRDH_1SDV_20190112T035232_20190112T0352...,2019-01-12 03:51:17+00:00,2019-01-12 03:52:59+00:00,2019-01-12T03:52:08Z,31.103129,31.596094,31.259670,...,2019-01-12 08:51:18+00:00,1.45,S1_20190112_035117_035259_VV_0||2019-01-12 08:...,dataset_out\sent1\S1_20190112_035117_035259_VV...,dataset_out\sentinel2\S1_20190112_035117_03525...,2019-01-12 03:51:17+00:00,2019-01-12 08:51:18+00:00,5.000278,True,True
2,S1_20190122_155641_155735_VV_0,ow-0015.jpg,ow-0015-01-000032,S1B_IW_GRDH_1SDV_20190122T155641_20190122T1557...,2019-01-22 15:56:41+00:00,2019-01-22 15:57:35+00:00,2019-01-22T15:57:08Z,31.160483,32.720001,31.533496,...,2019-01-22 08:51:04+00:00,3.89,S1_20190122_155641_155735_VV_0||2019-01-22 08:...,dataset_out\sent1\S1_20190122_155641_155735_VV...,dataset_out\sentinel2\S1_20190122_155641_15573...,2019-01-22 15:56:41+00:00,2019-01-22 08:51:04+00:00,-7.093611,True,True
3,S1_20190122_155641_155735_VV_2,ow-0016.jpg,ow-0016-01-000034,S1B_IW_GRDH_1SDV_20190122T155641_20190122T1557...,2019-01-22 15:56:41+00:00,2019-01-22 15:57:35+00:00,2019-01-22T15:57:08Z,30.195389,33.563524,30.361045,...,2019-01-22 08:50:56+00:00,3.23,S1_20190122_155641_155735_VV_2||2019-01-22 08:...,dataset_out\sent1\S1_20190122_155641_155735_VV...,dataset_out\sentinel2\S1_20190122_155641_15573...,2019-01-22 15:56:41+00:00,2019-01-22 08:50:56+00:00,-7.095833,True,True
4,S1_20190122_155641_155735_VV_3,ow-0017.jpg,ow-0017-03-000044,S1B_IW_GRDH_1SDV_20190122T155641_20190122T1557...,2019-01-22 15:56:41+00:00,2019-01-22 15:57:35+00:00,2019-01-22T15:57:08Z,29.951816,33.534557,30.113423,...,2019-01-22 08:50:56+00:00,3.23,S1_20190122_155641_155735_VV_3||2019-01-22 08:...,dataset_out\sent1\S1_20190122_155641_155735_VV...,dataset_out\sentinel2\S1_20190122_155641_15573...,2019-01-22 15:56:41+00:00,2019-01-22 08:50:56+00:00,-7.095833,True,True


In [8]:

BANDS_ALL = ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B11","B12"]  
BANDS = BANDS_ALL


In [9]:
import os
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from sentinelhub import SentinelHubRequest, MimeType

def _save_array_as_geotiff(arr, bbox, out_path, crs_epsg="EPSG:4326"):
    # arr can be (H,W), (H,W,1), (H,W,C)
    if arr.ndim == 2:
        H, W = arr.shape
        C = 1
    elif arr.ndim == 3:
        H, W, C = arr.shape
    else:
        raise ValueError(f"Unexpected array shape: {arr.shape}")

    minx, miny, maxx, maxy = bbox
    transform = from_bounds(minx, miny, maxx, maxy, W, H)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    profile = {
        "driver": "GTiff",
        "height": H,
        "width": W,
        "count": C,
        "dtype": str(arr.dtype),
        "crs": crs_epsg,
        "transform": transform,
    }

    with rasterio.open(out_path, "w", **profile) as dst:
        if C == 1:
            band = arr[:, :, 0] if arr.ndim == 3 else arr
            dst.write(band, 1)
        else:
            for i in range(C):
                dst.write(arr[:, :, i], i + 1)

def download_s2_patch_tiff(cfg, bbox, s2_dt, out_path, bands):
    t0 = (s2_dt - pd.Timedelta(minutes=2)).strftime("%Y-%m-%dT%H:%M:%SZ")
    t1 = (s2_dt + pd.Timedelta(minutes=2)).strftime("%Y-%m-%dT%H:%M:%SZ")

    evalscript = f"""
    //VERSION=3
    function setup() {{
      return {{
        input: [{{
          bands: {bands},
          units: "REFLECTANCE"
        }}],
        output: {{
          bands: {len(bands)},
          sampleType: "FLOAT32"
        }}
      }};
    }}
    function evaluatePixel(s) {{
      return [{", ".join([f"s.{b}" for b in bands])}];
    }}
    """

    res_m, (width, height) = pick_resolution_and_size(bbox, BASE_RESOLUTION_M, MAX_DIM_PX)

    req = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[SentinelHubRequest.input_data(
            data_collection=S2L2A_CDSE,
            time_interval=(t0, t1),
        )],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        bbox=bbox,
        size=(width, height),
        config=cfg,
    )

    arr = req.get_data()[0]  # (H,W,C)
    _save_array_as_geotiff(arr, bbox, out_path)

def download_s1_patch_tiff(cfg, bbox, t_from, t_to, out_path, pol="VV"):
    evalscript = f"""
    //VERSION=3
    function setup() {{
      return {{
        input: [{{
          bands: ["{pol}"],
          units: "LINEAR_POWER"
        }}],
        output: {{
          bands: 1,
          sampleType: "FLOAT32"
        }}
      }};
    }}
    function evaluatePixel(s) {{
      return [s.{pol}];
    }}
    """

    res_m, (width, height) = pick_resolution_and_size(bbox, BASE_RESOLUTION_M, MAX_DIM_PX)

    req = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[SentinelHubRequest.input_data(
            data_collection=S1IW_CDSE,
            time_interval=(t_from, t_to),
            other_args={"processing": {"orthorectify": "true"}}
        )],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        bbox=bbox,
        size=(width, height),
        config=cfg,
    )

    arr = req.get_data()[0]  # can be (H,W) or (H,W,1)
    # ensure float32 for stable GeoTIFF writing
    arr = arr.astype(np.float32) if hasattr(arr, "astype") else arr
    _save_array_as_geotiff(arr, bbox, out_path)


In [ ]:
MAX_WORKERS = 2
RETRIES = 2

def _download_one_pair(row):
    patch = str(row["patch_name"])

    bbox = BBox((row["min_lon"], row["min_lat"], row["max_lon"], row["max_lat"]), crs=CRS.WGS84)

    # filenames
    s1_dt = pd.to_datetime(row["sar_date_from"], utc=True)
    s2_dt = pd.to_datetime(row["s2_time"], utc=True)

    s1_out = os.path.join(S1_TIFF_DIR, f"{patch}__S1__{s1_dt.strftime('%Y%m%dT%H%M%S')}Z.tif")
    s2_out = os.path.join(S2_TIFF_DIR, f"{patch}__S2__{s2_dt.strftime('%Y%m%dT%H%M%S')}Z.tif")

    # time interval for SAR
    t_from = pd.to_datetime(row["sar_date_from"], utc=True).strftime("%Y-%m-%dT%H:%M:%SZ")
    t_to   = pd.to_datetime(row["sar_date_to"], utc=True).strftime("%Y-%m-%dT%H:%M:%SZ")

    
    if os.path.exists(s1_out) and os.path.exists(s2_out):
        return ("SKIP", patch)

    last_err = None
    for k in range(RETRIES):
        try:
            if not os.path.exists(s1_out):
                download_s1_patch_tiff(config, bbox, t_from, t_to, s1_out, pol="VV")
            if not os.path.exists(s2_out):
                download_s2_patch_tiff(config, bbox, s2_dt, s2_out, bands=BANDS)
            return ("OK", patch)
        except Exception as e:
            last_err = e
            time.sleep((2 ** k) * 0.6 + random.random() * 0.3)

    return ("FAILED", patch, str(last_err))

records = pairs.to_dict("records")
ok = skip = 0
failed = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(_download_one_pair, r) for r in records]
    for fut in tqdm(as_completed(futures), total=len(futures)):
        res = fut.result()
        if res[0] == "OK":
            ok += 1
        elif res[0] == "SKIP":
            skip += 1
        else:
            failed.append(res)

print(f"Done. OK={ok}, SKIP={skip}, FAILED={len(failed)}")
if failed:
    print("First 5 failures:")
    for x in failed[:5]:
        print(x)
print("S1 TIFF dir:", S1_TIFF_DIR)
print("S2 TIFF dir:", S2_TIFF_DIR)


  0%|          | 0/521 [00:00<?, ?it/s]

## sent3 download

In [1]:

# ----------------------------------------------------
import os
import pandas as pd
from datetime import timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

from sentinelhub import (
    SHConfig,
    DataCollection,
    SentinelHubCatalog,
    BBox,
    CRS,
)

# ----------------------------------------------------
# Configuration (Copernicus Data Space Ecosystem / Sentinel Hub)
# ----------------------------------------------------
CLIENT_ID = "sh-a2c7c0a9-4e39-44d9-9c98-e1ad1df57651"
CLIENT_SECRET = "p8v5PEnojoV1WUEtkd4Z70x5QGdya0QR"

config = SHConfig()
config.sh_client_id = CLIENT_ID
config.sh_client_secret = CLIENT_SECRET

# CDSE endpoints
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

# ----------------------------------------------------
# Settings
# ----------------------------------------------------
OUT_DIR = "dataset_out"
os.makedirs(OUT_DIR, exist_ok=True)

S3_MATCHES_CSV = os.path.join(OUT_DIR, "sent1_sent3.csv")
WINDOW_DAYS = 0.5  # 12 hours

# Use the built-in collection; config.sh_base_url will route requests to CDSE
S3_OLCI = DataCollection.SENTINEL3_OLCI


# ----------------------------------------------------
# Helpers
# ----------------------------------------------------
def sar_center_time(date_from, date_to):
    """Midpoint between two UTC timestamps."""
    return date_from + (date_to - date_from) / 2


def find_best_s3_match(catalog: SentinelHubCatalog, bbox: BBox, center_t: pd.Timestamp):
    """Search Sentinel-3 OLCI in a time window and return the closest acquisition by time."""
    center_t = pd.to_datetime(center_t, utc=True)

    start = (center_t - timedelta(days=WINDOW_DAYS)).strftime("%Y-%m-%dT%H:%M:%SZ")
    end = (center_t + timedelta(days=WINDOW_DAYS)).strftime("%Y-%m-%dT%H:%M:%SZ")

    # NOTE: Sentinel-3 items may or may not have eo:cloud_cover; keep it optional.
    search_iter = catalog.search(
        S3_OLCI,
        bbox=bbox,
        time=(start, end),
        fields={"include": ["id", "properties.datetime", "properties.eo:cloud_cover"], "exclude": []},
    )

    items = list(search_iter)
    if not items:
        return None

    def score(item):
        dt = pd.to_datetime(item["properties"]["datetime"], utc=True)
        return abs((dt - center_t).total_seconds())

    best = min(items, key=score)
    best_dt = pd.to_datetime(best["properties"]["datetime"], utc=True)
    best_cloud = best["properties"].get("eo:cloud_cover", None)

    return {
        "s3_id": best["id"],
        "s3_datetime": best_dt,
        "s3_cloud": best_cloud,
    }


def process_patches(
    csv_path="sar_unique.csv",
    out_csv=S3_MATCHES_CSV,
    max_workers=8,
    show_errors=True,
):
    """
    Reads SAR patch list, finds closest Sentinel-3 OLCI match per patch, writes output CSV.
    Expected columns in csv_path:
      patch_name, min_lon, min_lat, max_lon, max_lat, date_from, date_to
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)

    required = ["patch_name", "min_lon", "min_lat", "max_lon", "max_lat", "date_from", "date_to"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {csv_path}: {missing}")

    df["date_from"] = pd.to_datetime(df["date_from"], utc=True)
    df["date_to"] = pd.to_datetime(df["date_to"], utc=True)
    df["sar_center_time"] = df.apply(lambda r: sar_center_time(r["date_from"], r["date_to"]), axis=1)

    records = df.to_dict("records")
    s3_matches = []

    print(f"Searching Sentinel-3 matches for {len(records)} patches...")

    def _match(r):
        bbox = BBox((r["min_lon"], r["min_lat"], r["max_lon"], r["max_lat"]), crs=CRS.WGS84)
        center_t = pd.to_datetime(r["sar_center_time"], utc=True)

        # Create catalog inside thread (safe + avoids shared session issues)
        cat = SentinelHubCatalog(config=config)

        m = find_best_s3_match(cat, bbox, center_t)
        if not m:
            return None

        return {
            "patch_name": r["patch_name"],
            "sar_center_time": center_t.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "s3_id": m["s3_id"],
            "s3_datetime": m["s3_datetime"].strftime("%Y-%m-%dT%H:%M:%SZ"),
            "s3_cloud": m["s3_cloud"],
        }

    errors = 0
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(_match, r) for r in records]
        for fut in tqdm(as_completed(futures), total=len(futures)):
            try:
                res = fut.result()
                if res:
                    s3_matches.append(res)
            except Exception as e:
                errors += 1
                if show_errors:
                    print("Error:", repr(e))

    out_df = pd.DataFrame(s3_matches)
    out_df.to_csv(out_csv, index=False)

    print(f"Saved {len(out_df)} matches to {out_csv}")
    if errors:
        print(f"Finished with {errors} errors (see prints above).")

    return out_df


# ----------------------------------------------------
# Run (in a notebook cell)
# ----------------------------------------------------
out_df = process_patches(csv_path="sar_unique.csv", out_csv=S3_MATCHES_CSV, max_workers=8)
out_df.head()


Searching Sentinel-3 matches for 1181 patches...


  0%|          | 0/1181 [00:00<?, ?it/s]

Saved 1127 matches to dataset_out\sent1_sent3.csv


,patch_name,sar_center_time,s3_id,s3_datetime,s3_cloud
0,S1_20190104_155638_155818_VV_2,2019-01-04T15:57:28Z,S3A_OL_1_EFR____20190104T081617_20190104T08191...,2019-01-04T08:16:17Z,None
1,S1_20190111_154836_155016_VV_6,2019-01-11T15:49:26Z,S3B_OL_1_EFR____20190111T075530_20190111T07583...,2019-01-11T07:55:30Z,None
2,S1_20190119_034258_034438_VV_0,2019-01-19T03:43:48Z,S3B_OL_1_EFR____20190119T074801_20190119T07510...,2019-01-19T07:48:01Z,None
3,S1_20190101_034235_034350_VV_1,2019-01-01T03:43:12Z,S3A_OL_1_EFR____20190101T075350_20190101T07565...,2019-01-01T07:53:50Z,None
4,S1_20190112_153943_154058_VV_2,2019-01-12T15:40:20Z,S3A_OL_1_EFR____20190112T080849_20190112T08114...,2019-01-12T08:08:49Z,None


In [ ]:

import os
import re
import pandas as pd
from datetime import timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

from sentinelhub import (
    SHConfig, CRS, BBox, DataCollection,
    SentinelHubRequest, MimeType, MosaickingOrder,
    bbox_to_dimensions
)

# --- your credentials / endpoints ---
CLIENT_ID = "YOUR_CLIENT_ID"
CLIENT_SECRET = "YOUR_CLIENT_SECRET"

config = SHConfig()
config.sh_client_id = CLIENT_ID
config.sh_client_secret = CLIENT_SECRET
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"


# ============================================================
# 2) Inputs
#    - Put your 50 Sentinel-1 product IDs here (column: sentinel_id)
# ============================================================
S1_IDS = [
    "S1A_IW_GRDH_1SDV_20190504T155704_20190504T155729_027080_030D1A_1E63.SAFE",
    "S1A_IW_GRDH_1SDV_20190512T035208_20190512T035233_027189_0310AB_E893.SAFE",
    "S1B_IW_GRDH_1SDV_20190610T154813_20190610T154838_016636_01F4FD_EE80.SAFE",
    "S1B_IW_GRDH_1SDV_20190623T035134_20190623T035159_016818_01FA62_D8E1.SAFE",
    "S1A_IW_GRDH_1SDV_20190626T160517_20190626T160542_027853_032501_FC2B.SAFE",
    "S1A_IW_GRDH_1SDV_20190711T035146_20190711T035211_028064_032B5E_C727.SAFE",
    "S1A_IW_GRDH_1SDV_20190711T035211_20190711T035236_028064_032B5E_F1C5.SAFE",
    "S1A_IW_GRDH_1SDV_20190716T040022_20190716T040047_028137_032D90_2975.SAFE",
    "S1A_IW_GRDH_1SDV_20190718T034328_20190718T034353_028166_032E76_4818.SAFE",
    "S1A_IW_GRDH_1SDV_20190723T035147_20190723T035212_028239_0330A7_B840.SAFE",
    "S1A_IW_GRDH_1SDV_20190723T035212_20190723T035237_028239_0330A7_A061.SAFE",
    "S1A_IW_GRDH_1SDV_20190723T035237_20190723T035304_028239_0330A7_2013.SAFE",
    "S1B_IW_GRDH_1SDV_20190723T154038_20190723T154103_017263_020781_85CD.SAFE",
    "S1A_IW_GRDH_1SDV_20190730T034328_20190730T034353_028341_0333D2_4C47.SAFE",
    "S1A_IW_GRDH_1SDV_20190730T034353_20190730T034418_028341_0333D2_2D41.SAFE",
    "S1A_IW_GRDH_1SDV_20190820T155710_20190820T155735_028655_033E24_AFFA.SAFE",
    "S1B_IW_GRDH_1SDV_20190822T035137_20190822T035202_017693_021497_9677.SAFE",
    "S1B_IW_GRDH_1SDV_20190822T035047_20190822T035112_017693_021497_D54D.SAFE",
    "S1B_IW_GRDH_1SDV_20190824T033441_20190824T033506_017722_021577_8148.SAFE",
    "S1A_IW_GRDH_1SDV_20190825T160521_20190825T160546_028728_0340B1_6F50.SAFE",
    "S1A_IW_GRDH_1SDV_20190827T154908_20190827T154933_028757_0341B2_83CA.SAFE",
    "S1B_IW_GRDH_1SDV_20190914T154844_20190914T154909_018036_021F43_27F9.SAFE",
    "S1A_IW_GRDH_1SDV_20191019T155737_20191019T155802_029530_035C5B_E72A.SAFE",
    "S1A_IW_GRDH_1SDV_20191019T155712_20191019T155737_029530_035C5B_D123.SAFE",
    "S1B_IW_GRDH_1SDV_20191021T035114_20191021T035139_018568_022FC0_0980.SAFE",
    "S1A_IW_GRDH_1SDV_20190531T034350_20190531T034415_027466_031954_419E.SAFE",
    "S1B_IW_GRDH_1SDV_20190603T155613_20190603T155637_016534_01F1F4_6FF3.SAFE",
    "S1A_IW_GRDH_1SDV_20190730T034418_20190730T034443_028341_0333D2_9F38.SAFE",
    "S1B_IW_GRDH_1SDV_20190730T153223_20190730T153248_017365_020A74_A71C.SAFE",
    "S1B_IW_GRDH_1SDV_20190810T035137_20190810T035202_017518_020F20_E49F.SAFE",
    "S1A_IW_GRDH_1SDV_20190921T035125_20190921T035150_029114_034E08_0160.SAFE",
    "S1B_IW_GRDH_1SDV_20191130T155618_20191130T155643_019159_02429A_7E10.SAFE",
]


# CSV paths (use local names; fallback to /mnt/data if you run in this environment)
SAR_UNIQUE_PATH = "sar_unique.csv"
MATCHES_PATH = os.path.join("dataset_out", "sent1_sent3.csv")

if not os.path.exists(SAR_UNIQUE_PATH) and os.path.exists("/mnt/data/sar_unique.csv"):
    SAR_UNIQUE_PATH = "/mnt/data/sar_unique.csv"
if not os.path.exists(MATCHES_PATH) and os.path.exists("/mnt/data/sent1_sent3.csv"):
    MATCHES_PATH = "/mnt/data/sent1_sent3.csv"

OUT_DIR = "dataset_out"
SENT3_DIR = os.path.join(OUT_DIR, "sent3")
os.makedirs(SENT3_DIR, exist_ok=True)

# Download settings
RESOLUTION_M = 300
TIME_WINDOW_MIN = 20           # +/- minutes around matched s3_datetime
MAX_WORKERS = 3                # keep low to avoid API throttling


# ============================================================
# 3) DataCollections
# ============================================================
S3_OLCI_L1 = DataCollection.SENTINEL3_OLCI.define_from("s3_olci_l1", service_url=config.sh_base_url)

# OLCI L2 is available in CDSE; sentinelhub-py may not have a predefined enum -> define it:
S3_OLCI_L2 = DataCollection.define(
    name="SENTINEL3_OLCI_L2",
    api_id="sentinel-3-olci-l2",
    catalog_id="sentinel-3-olci-l2",
    service_url=config.sh_base_url
)


# ============================================================
# 4) Evalscripts (L1 reflectance + QA) and (L2 water products + reflectance)
# ============================================================
EVALSCRIPT_L1 = r"""
//VERSION=3
function setup() {
  return {
    input: [{
      bands: [
        "B01","B02","B03","B04","B05","B06","B07","B08","B09","B10","B11","B12",
        "B13","B14","B15","B16","B17","B18","B19","B20","B21",
        "dataMask","QUALITY_FLAGS"
      ]
    }],
    output: [
      { id: "ref", bands: 22, sampleType: "FLOAT32" },   // B01..B21 + dataMask
      { id: "qa",  bands: 1,  sampleType: "UINT32" }     // QUALITY_FLAGS
    ]
  }
}

function evaluatePixel(s) {
  return {
    ref: [
      s.B01,s.B02,s.B03,s.B04,s.B05,s.B06,s.B07,s.B08,s.B09,s.B10,s.B11,s.B12,
      s.B13,s.B14,s.B15,s.B16,s.B17,s.B18,s.B19,s.B20,s.B21,
      s.dataMask
    ],
    qa: [s.QUALITY_FLAGS]
  };
}
"""

# L2 WATER bands: water products + optical reflectance that exists in L2 (some bands missing)
EVALSCRIPT_L2_WATER = r"""
//VERSION=3
function setup() {
  return {
    input: [{
      bands: [
        // water products
        "CHL_OC4ME","CHL_NN","TSM_NN","KD490_M07","ADG443_NN","A865","T865","PAR","IWV_W",
        // reflectance bands available in L2 water (note: B13, B14, B15, B19, B20 are not available)
        "B01","B02","B03","B04","B05","B06","B07","B08","B09","B10","B11","B12",
        "B16","B17","B18","B21",
        "dataMask"
      ]
    }],
    output: [
      { id: "water", bands: 10, sampleType: "FLOAT32" },  // water products + dataMask
      { id: "ref",   bands: 17, sampleType: "FLOAT32" }   // reflectance subset + dataMask
    ]
  };
}

function evaluatePixel(s) {
  return {
    water: [
      s.CHL_OC4ME, s.CHL_NN, s.TSM_NN, s.KD490_M07, s.ADG443_NN, s.A865, s.T865, s.PAR, s.IWV_W,
      s.dataMask
    ],
    ref: [
      s.B01,s.B02,s.B03,s.B04,s.B05,s.B06,s.B07,s.B08,s.B09,s.B10,s.B11,s.B12,
      s.B16,s.B17,s.B18,s.B21,
      s.dataMask
    ]
  };
}
"""


# ============================================================
# 5) Build the download plan (filter by your S1 IDs -> join to S3 matches)
# ============================================================
def build_plan(s1_ids):
    if not s1_ids:
        raise ValueError("S1_IDS is empty. Paste your 50 Sentinel-1 product IDs into S1_IDS.")

    sar = pd.read_csv(SAR_UNIQUE_PATH)
    m = pd.read_csv(MATCHES_PATH)

    # filter SAR rows by given Sentinel-1 ids
    sar_sel = sar[sar["sentinel_id"].isin(s1_ids)].copy()

    # join with S1->S3 match table using patch_name
    plan = sar_sel.merge(m, on="patch_name", how="inner")

    # normalize datetimes
    plan["s3_datetime"] = pd.to_datetime(plan["s3_datetime"], utc=True)

    # keep useful cols
    keep = [
        "patch_name",
        "sentinel_id",
        "min_lon","min_lat","max_lon","max_lat",
        "s3_id","s3_datetime"
    ]
    plan = plan[keep].drop_duplicates().reset_index(drop=True)

    plan_path = os.path.join(SENT3_DIR, "download_plan.csv")
    plan.to_csv(plan_path, index=False)
    print(f"Plan rows: {len(plan)}  |  saved: {plan_path}")
    return plan


# ============================================================
# 6) Downloader helpers
# ============================================================
def safe_name(s: str) -> str:
    s = re.sub(r"[^\w\-\.]+", "_", str(s))
    return s[:180]

def iso_z(dt: pd.Timestamp) -> str:
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

def make_bbox(row) -> BBox:
    return BBox((row.min_lon, row.min_lat, row.max_lon, row.max_lat), crs=CRS.WGS84)

def make_time_interval(center_dt: pd.Timestamp):
    start = center_dt - timedelta(minutes=TIME_WINDOW_MIN)
    end = center_dt + timedelta(minutes=TIME_WINDOW_MIN)
    return (iso_z(start), iso_z(end))

def request_l1(bbox: BBox, size, time_interval, data_folder: str):
    return SentinelHubRequest(
        evalscript=EVALSCRIPT_L1,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=S3_OLCI_L1,
                time_interval=time_interval,
                mosaicking_order=MosaickingOrder.MOST_RECENT,
            )
        ],
        responses=[
            SentinelHubRequest.output_response("ref", MimeType.TIFF),
            SentinelHubRequest.output_response("qa",  MimeType.TIFF),
        ],
        bbox=bbox,
        size=size,
        data_folder=data_folder,
        config=config,
    )

def request_l2_water(bbox: BBox, size, time_interval, data_folder: str):
    return SentinelHubRequest(
        evalscript=EVALSCRIPT_L2_WATER,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=S3_OLCI_L2,
                time_interval=time_interval,
                mosaicking_order=MosaickingOrder.MOST_RECENT,
                # You can also use max_cloud_coverage here for L2:
                # max_cloud_coverage=30
            )
        ],
        responses=[
            SentinelHubRequest.output_response("water", MimeType.TIFF),
            SentinelHubRequest.output_response("ref",   MimeType.TIFF),
        ],
        bbox=bbox,
        size=size,
        data_folder=data_folder,
        config=config,
    )

def run_request_and_rename(req: SentinelHubRequest, rename_map: dict):
    """
    rename_map: { "ref": "my_ref.tif", "qa": "my_qa.tif", ... }
    """
    # download + save
    req.get_data(save_data=True)

    # sentinelhub-py stores files using internal names; we rename them to stable names
    # filenames are accessible via get_filename_list()
    saved_files = req.get_filename_list()

    # Build index from response id to file path (works with typical sentinelhub-py naming)
    # If your sentinelhub-py version uses different naming, you can just print(saved_files) once.
    for rel in saved_files:
        base = os.path.basename(rel)
        # try to detect which response it is by containing "_<id>." or "<id>."
        for rid, new_name in rename_map.items():
            if (f"_{rid}." in base) or base.startswith(f"{rid}.") or (f"{rid}." in base):
                src = os.path.join(req.data_folder, rel)
                dst = os.path.join(req.data_folder, new_name)
                if os.path.exists(src):
                    os.replace(src, dst)
                break


# ============================================================
# 7) Download all matched Sentinel-3 patches (L1 + L2) into dataset_out/sent3/
# ============================================================
def download_all(plan: pd.DataFrame, max_workers=MAX_WORKERS):
    results = []
    errors = 0

    def _one(row):
        row = row.copy()
        patch = safe_name(row["patch_name"])
        s1id = safe_name(row["sentinel_id"])
        dt = pd.to_datetime(row["s3_datetime"], utc=True)

        patch_dir = os.path.join(SENT3_DIR, patch)
        os.makedirs(patch_dir, exist_ok=True)

        bbox = make_bbox(row)
        size = bbox_to_dimensions(bbox, resolution=RESOLUTION_M)
        t_int = make_time_interval(dt)

        stamp = dt.strftime("%Y%m%dT%H%M%SZ")

        # ---- L1 (reflectance + QA flags) ----
        req1 = request_l1(bbox, size, t_int, patch_dir)
        run_request_and_rename(
            req1,
            rename_map={
                "ref": f"{patch}__{stamp}__L1_reflectance_B01-B21+mask.tif",
                "qa":  f"{patch}__{stamp}__L1_quality_flags.tif",
            },
        )

        # ---- L2 (water products + reflectance subset) ----
        req2 = request_l2_water(bbox, size, t_int, patch_dir)
        run_request_and_rename(
            req2,
            rename_map={
                "water": f"{patch}__{stamp}__L2_water_products+mask.tif",
                "ref":   f"{patch}__{stamp}__L2_reflectance_subset+mask.tif",
            },
        )

        # save small manifest line
        return {
            "patch_name": row["patch_name"],
            "sentinel_id": row["sentinel_id"],
            "s3_id": row["s3_id"],
            "s3_datetime": iso_z(dt),
            "out_dir": patch_dir,
        }

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = [ex.submit(_one, plan.iloc[i]) for i in range(len(plan))]
        for fut in tqdm(as_completed(futs), total=len(futs)):
            try:
                results.append(fut.result())
            except Exception as e:
                errors += 1
                print("Download error:", repr(e))

    manifest = pd.DataFrame(results)
    manifest_path = os.path.join(SENT3_DIR, "download_manifest.csv")
    manifest.to_csv(manifest_path, index=False)
    print(f"Done. Downloaded: {len(manifest)} patches | errors: {errors}")
    print(f"Manifest: {manifest_path}")
    return manifest


# ============================================================
# 8) RUN
# ============================================================
plan_df = build_plan(S1_IDS)
manifest_df = download_all(plan_df, max_workers=MAX_WORKERS)
manifest_df.head()


Plan rows: 0  |  saved: dataset_out\sent3\download_plan.csv


0it [00:00, ?it/s]

Done. Downloaded: 0 patches | errors: 0
Manifest: dataset_out\sent3\download_manifest.csv


""
